In [24]:
from __future__ import annotations

from pathlib import Path


def resolve_validation_dir() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "app.py").exists() and (p / "statistical_detector.py").exists():
            return p
    raise RuntimeError("Run this notebook from the validation service directory (services/validation).")


def resolve_hackathon_2026_dir(validation_dir: Path) -> Path:
    for p in [validation_dir, *validation_dir.parents]:
        if (p / "dataset" / "generate.py").exists():
            return p
    raise RuntimeError("Could not find Hackathon_2026 root (missing dataset/generate.py)")


VALIDATION_DIR = resolve_validation_dir()
HACKATHON_2026_DIR = resolve_hackathon_2026_dir(VALIDATION_DIR)

DATASET_DIR = HACKATHON_2026_DIR / "dataset"
DATASET_OUTPUT_DIR = DATASET_DIR / "output_ml"
MANIFEST_PATH = DATASET_OUTPUT_DIR / "dataset_manifest.json"

VENV_DIR = VALIDATION_DIR / ".venv-ml"
VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"

print("VALIDATION_DIR=", VALIDATION_DIR)
print("HACKATHON_2026_DIR=", HACKATHON_2026_DIR)
print("DATASET_DIR=", DATASET_DIR)
print("MANIFEST_PATH=", MANIFEST_PATH)
print("VENV_PYTHON=", VENV_PYTHON)


VALIDATION_DIR= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\services\validation
HACKATHON_2026_DIR= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026
DATASET_DIR= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset
MANIFEST_PATH= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\output_ml\dataset_manifest.json


In [23]:
from __future__ import annotations

import subprocess
import sys
import venv

# Create a clean venv for ML (avoids Conda TF binary conflicts)
SETUP_VENV = True

if SETUP_VENV:
    ml_reqs = VALIDATION_DIR / "requirements.notebook.txt"  # pinned ML stack
    dataset_reqs = HACKATHON_2026_DIR / "dataset" / "requirements.txt"  # generator deps

    print("current_python=", sys.executable)
    print("VENV_DIR=", VENV_DIR)
    print("VENV_PYTHON=", VENV_PYTHON)
    print("ml_reqs=", ml_reqs)
    print("dataset_reqs=", dataset_reqs)

    missing_files = [p for p in [ml_reqs, dataset_reqs] if not p.exists()]
    assert not missing_files, f"requirements.txt not found: {missing_files}"

    if not VENV_PYTHON.exists():
        venv.EnvBuilder(with_pip=True).create(VENV_DIR)

    def run(args: list[str]) -> None:
        proc = subprocess.run(args, capture_output=True, text=True)
        if proc.returncode != 0:
            print("\n--- stdout ---\n", proc.stdout)
            print("\n--- stderr ---\n", proc.stderr)
            raise RuntimeError(f"command failed (exit={proc.returncode}) for: {' '.join(args)}")

    run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"])
    run([str(VENV_PYTHON), "-m", "pip", "install", "--no-cache-dir", "-r", str(ml_reqs)])
    run([str(VENV_PYTHON), "-m", "pip", "install", "-r", str(dataset_reqs)])

    # We TRY to register a kernel, but Cursor/VSCode sometimes won't list it.
    # Training below will work even without switching kernels.
    run([
        str(VENV_PYTHON),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        "hackathon-validation-ml",
        "--display-name",
        "Hackathon Validation ML",
    ])

    print("Venv ready.")
    print("If you don't see the kernel, you can still train via subprocess using VENV_PYTHON.")


python= c:\Users\waelc\anaconda3\envs\tensorflow\python.exe
ml_reqs= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\services\validation\requirements.notebook.txt
dataset_reqs= C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\requirements.txt

--- pip stdout ---
  Obtaining dependency information for numpy==1.26.4 from https://files.pythonhosted.org/packages/3f/6b/5610004206cf7f8e7ad91c5a85a8c71b2f2f8051a0c0c4d5916b76d6cbb2/numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 61.0/61.0 kB 3.2 MB/s eta 0:00:00
  Obtaining dependency information for scikit-learn==1.4.2 from https://files.pythonhosted.org/packages/79/3d/02d5d3ed359498fec3abdf65407d3c07e3b8765af17464969055aaec5171/scikit_learn-1.4.2-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for joblib==1.4.2 from https://files.pythonhosted.org/packages/91/29/df4b9b42f2be0b623cbd5e21

RuntimeError: pip failed (exit=1) for: c:\Users\waelc\anaconda3\envs\tensorflow\python.exe -m pip install --upgrade --force-reinstall --no-cache-dir -r C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\services\validation\requirements.notebook.txt

In [ ]:
from __future__ import annotations

import importlib


def missing_modules(mods: list[str]) -> list[str]:
    missing: list[str] = []
    for m in mods:
        try:
            importlib.import_module(m)
        except Exception:
            missing.append(m)
    return missing


required = ["tqdm", "reportlab", "faker"]
missing = missing_modules(required)
print("missing=", missing)
assert not missing, f"Missing generator deps: {missing} (install dataset/requirements.txt in your env)"


missing= []


In [ ]:
from __future__ import annotations

import runpy
import sys

N_PER_TYPE = 50
SCENARIOS = "coherent"
SEED = 42
FORCE_REGENERATE = False

DATASET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if FORCE_REGENERATE and MANIFEST_PATH.exists():
    MANIFEST_PATH.unlink()

if not MANIFEST_PATH.exists():
    sys.argv = [
        "generate.py",
        "--n",
        str(N_PER_TYPE),
        "--output",
        str(DATASET_OUTPUT_DIR),
        "--scenarios",
        SCENARIOS,
        "--seed",
        str(SEED),
    ]
    runpy.run_path(str(DATASET_DIR / "generate.py"), run_name="__main__")

print("manifest_exists=", MANIFEST_PATH.exists())
print("manifest_size_bytes=", MANIFEST_PATH.stat().st_size if MANIFEST_PATH.exists() else None)


═══════════════════════════════════════════
  Génération du dataset — 50 docs/type
  Sortie : C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\output_ml
═══════════════════════════════════════════

📄 FACTURE (50 documents)


  facture: 100%|██████████████████████| 50/50 [00:01<00:00, 29.41it/s]



📄 DEVIS (50 documents)


  devis: 100%|████████████████████████| 50/50 [00:01<00:00, 31.85it/s]



📄 ATTESTATION_URSSAF (50 documents)


...


📄 KBIS (50 documents)


  kbis: 100%|█████████████████████████| 50/50 [00:05<00:00,  8.97it/s]



📄 RIB (50 documents)


  rib: 100%|██████████████████████████| 50/50 [00:00<00:00, 63.21it/s]



═══════════════════════════════════════════
  ✅ 300 documents générés
  📁 PDF bruts :   C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\output_ml\raw
  📁 Dégradés :    C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\output_ml\noisy
  📁 Labels :      C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\output_ml\labels
  📋 Manifeste :   C:\Users\waelc\Desktop\Ipssi\Hackathon\Hackathon_2026\dataset\output_ml\dataset_manifest.json

  Par scénario :
    coherent     : 300

  Par type :
    facture              : 50
    devis                : 50
    attestation_urssaf   : 50
    attestation_siret    : 50
    kbis                 : 50
    rib                  : 50
═══════════════════════════════════════════
manifest_exists= True
manifest_size_bytes= 203180


In [ ]:
from __future__ import annotations

import json

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

factures_coherent = [
    d for d in manifest if d.get("type") == "facture" and d.get("scenario") == "coherent"
]

historical_docs = [{"entities": d.get("expected_fields", {})} for d in factures_coherent]

print("manifest_total=", len(manifest))
print("factures_coherent=", len(factures_coherent))
print("historical_docs=", len(historical_docs))
assert len(historical_docs) >= 10, "Need at least 10 coherent invoices to train"


manifest_total= 300
factures_coherent= 50
historical_docs= 50


In [ ]:
from __future__ import annotations

import json
import subprocess
import textwrap

assert "factures_coherent" in globals(), "Run cells 0→4 first"
assert VENV_PYTHON.exists(), "Run the venv setup cell first"

sample_entities = factures_coherent[0]["expected_fields"]

code = textwrap.dedent(
    f"""
    import json
    import sys
    from pathlib import Path

    validation_dir = Path({str(VALIDATION_DIR)!r})
    manifest_path = Path({str(MANIFEST_PATH)!r})

    sys.path.insert(0, str(validation_dir))
    import statistical_detector as sd

    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    factures = [d for d in manifest if d.get('type') == 'facture' and d.get('scenario') == 'coherent']
    historical_docs = [{{'entities': d.get('expected_fields', {{}})}} for d in factures]

    detector = sd.StatDetector()
    detector.fit(historical_docs)

    print('trained=', detector.trained)
    print('model_path=', sd.MODEL_PATH)
    print('model_exists=', sd.MODEL_PATH.exists())

    normal_doc = {{'type': 'facture', 'entities': json.loads({json.dumps(sample_entities)!r})}}
    extreme_doc = {{'type': 'facture', 'entities': dict(normal_doc['entities'])}}
    extreme_doc['entities']['montant_ht'] = float(extreme_doc['entities'].get('montant_ht') or 0) * 50
    extreme_doc['entities']['tva'] = float(extreme_doc['entities'].get('tva') or 0) * 50
    extreme_doc['entities']['montant_ttc'] = float(extreme_doc['entities'].get('montant_ttc') or 0) * 50

    print('normal_pred=', detector.predict(normal_doc))
    print('extreme_pred=', detector.predict(extreme_doc))
    """
)

proc = subprocess.run([str(VENV_PYTHON), "-c", code], capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"Training failed (exit={proc.returncode})")


AttributeError: module 'numpy._globals' has no attribute '_signature_descriptor'

ImportError: numpy._core.multiarray failed to import

In [ ]:
# Predictions are printed by the training cell (runs inside the clean venv).
# Keep this cell empty on purpose.


NameError: name 'factures_coherent' is not defined